In [8]:

from pathlib import Path

# Folder containing your original cricket JSON files
DATA_FOLDER = Path.home() / "Downloads" / "recently_added_2_json"

# Folder containing the text documents created in Part 3
DOCUMENTS_FOLDER = Path("data")

print("JSON folder exists:", DATA_FOLDER.exists())
print("Documents folder exists:", DOCUMENTS_FOLDER.exists())


JSON folder exists: True
Documents folder exists: True


In [3]:
uv add jq

Note: you may need to restart the kernel to use updated packages.


C:\Users\DELL\Documents\python\rag\cricket\.venv\Scripts\python.exe: No module named uv


In [4]:
import jq
print("jq installed successfully")

jq installed successfully


In [9]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Find all generated match documents
document_files = list(DOCUMENTS_FOLDER.glob("*.txt"))

print("Match documents found:", len(document_files))

# Configure the chunking method
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Read each document and split it
chunks = []

for document_file in document_files:
    text = document_file.read_text(encoding="utf-8")

    document_chunks = splitter.split_text(text)

    for chunk_number, chunk_text in enumerate(document_chunks):
        chunks.append({
            "match_id": document_file.stem,
            "chunk_id": chunk_number,
            "text": chunk_text
        })

print("Total chunks created:", len(chunks))

# Preview one chunk
if chunks:
    print("\nFirst chunk:")
    print(chunks[0]["text"])
else:
    print("No chunks created. Check the documents folder.")


Match documents found: 52


Total chunks created: 2223

First chunk:
Match ID: 1525659
Match Type: Test
Teams: Australia, South Africa
Date: 2026-10-09, 2026-10-10, 2026-10-11, 2026-10-12, 2026-10-13


In [10]:

import json

CHUNKS_FILE = Path("chunks.json")

with CHUNKS_FILE.open("w", encoding="utf-8") as file:
    json.dump(chunks, file, ensure_ascii=False, indent=2)

print("Chunks saved to:", CHUNKS_FILE.resolve())
print("Chunks saved:", len(chunks))


Chunks saved to: C:\Users\DELL\Documents\python\rag\cricket\cricket_analysis\chunks.json
Chunks saved: 2223


In [11]:

from langchain_ollama import OllamaEmbeddings

# Load the local embedding model
embeddings_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Embedding model loaded.")


Embedding model loaded.


In [12]:

# Get the text from the first chunk
sample_text = chunks[0]["text"]

# Convert the text into a vector
sample_embedding = embeddings_model.embed_query(sample_text)

print("Embedding generated successfully.")
print("Vector length:", len(sample_embedding))
print("First 10 values:", sample_embedding[:10])


Embedding generated successfully.
Vector length: 768
First 10 values: [-0.047357947, -0.021843102, -0.17896202, -0.03541704, 0.07712259, -0.0072995564, 0.049928375, -0.05270851, 0.031708706, 0.004110207]


In [13]:

from langchain_chroma import Chroma
from pathlib import Path

# Create or connect to the persistent ChromaDB
CHROMA_FOLDER = Path("chroma_db")
CHROMA_FOLDER.mkdir(parents=True, exist_ok=True)

db = Chroma(
    collection_name="cricket_matches",
    embedding_function=embeddings_model,
    persist_directory=str(CHROMA_FOLDER),
)

# Convert chunks into LangChain Document objects
from langchain_core.documents import Document

chunk_documents = [
    Document(
        page_content=chunk_item["text"],
        metadata={
            "match_id": str(chunk_item["match_id"]),
            "chunk_id": int(chunk_item["chunk_id"]),
        },
        id=f"{chunk_item['match_id']}_{chunk_item['chunk_id']}",
    )
    for chunk_item in chunks
]

# Index chunks in batches
batch_size = 100

for start in range(0, len(chunk_documents), batch_size):
    batch = chunk_documents[start:start + batch_size]
    db.add_documents(batch)

    print(
        f"Indexed {min(start + batch_size, len(chunk_documents))}"
        f" / {len(chunk_documents)} chunks"
    )

print("Total chunks indexed:", db._collection.count())


Indexed 100 / 2223 chunks
Indexed 200 / 2223 chunks
Indexed 300 / 2223 chunks
Indexed 400 / 2223 chunks
Indexed 500 / 2223 chunks
Indexed 600 / 2223 chunks
Indexed 700 / 2223 chunks
Indexed 800 / 2223 chunks
Indexed 900 / 2223 chunks
Indexed 1000 / 2223 chunks
Indexed 1100 / 2223 chunks
Indexed 1200 / 2223 chunks
Indexed 1300 / 2223 chunks
Indexed 1400 / 2223 chunks
Indexed 1500 / 2223 chunks
Indexed 1600 / 2223 chunks
Indexed 1700 / 2223 chunks
Indexed 1800 / 2223 chunks
Indexed 1900 / 2223 chunks
Indexed 2000 / 2223 chunks
Indexed 2100 / 2223 chunks
Indexed 2200 / 2223 chunks
Indexed 2223 / 2223 chunks
Total chunks indexed: 2223


In [2]:
from pathlib import Path
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

# Use the same folder you used yesterday
CHROMA_FOLDER = Path("chroma_db")

# Recreate the embedding model connection
embeddings_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Reopen the existing database
db = Chroma(
    collection_name="cricket_matches",
    embedding_function=embeddings_model,
    persist_directory=str(CHROMA_FOLDER),
)

print("Database reconnected!")
print("Chunks stored:", db._collection.count())

Database reconnected!
Chunks stored: 2223


In [3]:

question = "Which batters scored runs in the match?"

# Method 1: Similarity search
similarity_results = db.similarity_search(
    question,
    k=5,
)

# Method 2: Maximum Marginal Relevance (MMR)
mmr_results = db.max_marginal_relevance_search(
    question,
    k=5,
    fetch_k=15,
    lambda_mult=0.7,
)

print("SIMILARITY SEARCH")

for i, doc in enumerate(similarity_results, start=1):
    print(f"\n{i}. Match {doc.metadata.get('match_id')}:")
    print(doc.page_content[:200])


print("\n" + "=" * 50)
print("MMR SEARCH")

for i, doc in enumerate(mmr_results, start=1):
    print(f"\n{i}. Match {doc.metadata.get('match_id')}:")
    print(doc.page_content[:200])


SIMILARITY SEARCH

1. Match 1544848:
Innings 1: South Australia
Over 0, ball 1: MW Harvey faced F O'Neill; batter runs 0; extras 0; total runs 0.
Over 0, ball 2: MW Harvey faced F O'Neill; batter runs 0; extras 0; total runs 0.
Over 0, b

2. Match 1544849:
Innings 1: Tasmania
Over 0, ball 1: CP Jewell faced SA Abbott; batter runs 0; extras 0; total runs 0.
Over 0, ball 2: CP Jewell faced SA Abbott; batter runs 0; extras 0; total runs 0.
Over 0, ball 3: 

3. Match 1528717:
Innings 2: Australia
Over 0, ball 1: G Voll faced Marufa Akter; batter runs 1; extras 0; total runs 1.
Over 0, ball 2: P Litchfield faced Marufa Akter; batter runs 0; extras 0; total runs 0.
Over 0, b

4. Match 1554417:
Innings 2: Oman
Over 0, ball 1: Jatinder Singh faced NK Yadav; batter runs 0; extras 0; total runs 0.
Over 0, ball 2: Jatinder Singh faced NK Yadav; batter runs 0; extras 0; total runs 0.
Over 0, ball

5. Match 1528716:
Innings 2: Australia
Over 0, ball 1: G Voll faced Marufa Akter; batter runs 0; extr

In [7]:

# Use the results from your similarity search
retrieved_docs = similarity_results

# Build context with source information
context_parts = []

for i, doc in enumerate(retrieved_docs, start=1):
    match_id = doc.metadata.get("match_id", "Unknown")

    context_parts.append(
        f"[Source {i} | Match ID: {match_id}]\n"
        f"{doc.page_content}"
    )

context = "\n\n".join(context_parts)

print("Context created successfully.")
print("Number of source chunks:", len(retrieved_docs))
print("\nContext preview:\n")
print(context[:2500])


Context created successfully.
Number of source chunks: 5

Context preview:

[Source 1 | Match ID: 1544848]
Innings 1: South Australia
Over 0, ball 1: MW Harvey faced F O'Neill; batter runs 0; extras 0; total runs 0.
Over 0, ball 2: MW Harvey faced F O'Neill; batter runs 0; extras 0; total runs 0.
Over 0, ball 3: MW Harvey faced F O'Neill; batter runs 0; extras 0; total runs 0.
Over 0, ball 4: MW Harvey faced F O'Neill; batter runs 2; extras 0; total runs 2.
Over 0, ball 5: MW Harvey faced F O'Neill; batter runs 1; extras 0; total runs 1.
Over 0, ball 6: J Fraser-McGurk faced F O'Neill; batter runs 1; extras 0; total runs 1.
Over 1, ball 1: J Fraser-McGurk faced A Anlezark; batter runs 1; extras 0; total runs 1.
Over 1, ball 2: MW Harvey faced A Anlezark; batter runs 4; extras 0; total runs 4.
Over 1, ball 3: MW Harvey faced A Anlezark; batter runs 4; extras 0; total runs 4.
Over 1, ball 4: MW Harvey faced A Anlezark; batter runs 0; extras 0; total runs 0.
Over 1, ball 5: MW Harvey face

In [8]:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a cricket data assistant.

Answer the user's question using ONLY the context below.

Rules:
1. Do not invent players, scores, match results, or statistics.
2. If the context does not contain enough information, say:
   "The retrieved match documents do not contain enough
   information to answer this question."
3. Cite the relevant Match ID for every factual claim.
4. Clearly distinguish what the context states from what
   cannot be verified.
5. Do not claim that the retrieved chunks represent every
   match in the dataset.

Context:
{context}

Question:
{question}

Answer with evidence and Match ID citations:
""")

print("RAG prompt created successfully.")


RAG prompt created successfully.


In [9]:

question = "Which batters scored runs in the match?"

messages = prompt.format_messages(
    context=context,
    question=question,
)

print(messages[0].content[:3000])



You are a cricket data assistant.

Answer the user's question using ONLY the context below.

Rules:
1. Do not invent players, scores, match results, or statistics.
2. If the context does not contain enough information, say:
   "The retrieved match documents do not contain enough
   information to answer this question."
3. Cite the relevant Match ID for every factual claim.
4. Clearly distinguish what the context states from what
   cannot be verified.
5. Do not claim that the retrieved chunks represent every
   match in the dataset.

Context:
[Source 1 | Match ID: 1544848]
Innings 1: South Australia
Over 0, ball 1: MW Harvey faced F O'Neill; batter runs 0; extras 0; total runs 0.
Over 0, ball 2: MW Harvey faced F O'Neill; batter runs 0; extras 0; total runs 0.
Over 0, ball 3: MW Harvey faced F O'Neill; batter runs 0; extras 0; total runs 0.
Over 0, ball 4: MW Harvey faced F O'Neill; batter runs 2; extras 0; total runs 2.
Over 0, ball 5: MW Harvey faced F O'Neill; batter runs 1; extras

In [10]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

api_key = os.getenv("API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found. Check your .env file.")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=api_key
)

print("Groq LLM connected successfully.")

Groq LLM connected successfully.


In [13]:
from langchain_community.document_compressors import FlashrankRerank

reranker = FlashrankRerank(
    top_n=5,
    model="ms-marco-MiniLM-L-12-v2"
)

print("Reranker is ready!")

C:\Users\DELL\AppData\Local\Temp\ipykernel_3824\539447020.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_compressors import FlashrankRerank


Reranker is ready!


In [11]:
def ask_cricket_question(question, match_id=None):
    # 1. Retrieve relevant chunks
    search_filter = (
        {"match_id": str(match_id)}
        if match_id is not None
        else None
    )

    retrieved_docs = db.similarity_search(
        question,
        k=10,
        filter=search_filter
    )

    if not retrieved_docs:
        return {
            "answer": "No relevant cricket documents were found.",
            "sources": []
        }

    # 2. Rerank the retrieved chunks
    reranked_docs = reranker.compress_documents(
        retrieved_docs,
        query=question
    )

    if not reranked_docs:
        return {
            "answer": "No relevant evidence was found after reranking.",
            "sources": []
        }

    # 3. Build context with source IDs
    context = "\n\n".join(
        f"[Match ID: {doc.metadata.get('match_id', 'Unknown')}]\n"
        f"{doc.page_content}"
        for doc in reranked_docs
    )

    # 4. Generate an answer
    response = (prompt | llm).invoke({
        "context": context,
        "question": question
    })

    # 5. Return answer and source IDs
    sources = list(dict.fromkeys(
        str(doc.metadata.get("match_id", "Unknown"))
        for doc in reranked_docs
    ))

    return {
        "answer": response.content,
        "sources": sources
    }


In [17]:
result = ask_cricket_question(
    "Which players were dismissed, and how?" 
)

print("ANSWER:\n", result["answer"])
print("\nSOURCE MATCH IDs:", result["sources"])

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


ANSWER:
 **Dismissed players and their modes of dismissal**

| Player | Dismissal type | Match ID |
|--------|----------------|----------|
| CJ Rocchiccioli | caught | 1544866 |
| XC Bartlett | caught | 1544866 |
| MJ Swepson | caught | 1544866 |
| Arjun Kumal | caught | 1554416 |
| Mujibur Ali | bowled | 1554417 |
| C Chatphaisan | bowled | 1554549 |

**Evidence**

- Match 1544866 lists “Wicket: CJ Rocchiccioli, dismissal: caught”【1544866】, “Wicket: XC Bartlett, dismissal: caught”【1544866】, and “Wicket: MJ Swepson, dismissal: caught”【1544866】.  
- Match 1554416 lists “Wicket: Arjun Kumal, dismissal: caught”【1554416】.  
- Match 1554417 lists “Wicket: Mujibur Ali, dismissal: bowled”【1554417】.  
- Match 1554549 lists “Wicket: C Chatphaisan, dismissal: bowled”【1554549】.

SOURCE MATCH IDs: ['1544866', '1554416', '1554417', '1554549']
